## Single molecule tracking pipeline, python version
This script uploads a ready-to-use mask, as generated in omnipose, to database. To proceed, run the cells below by clicking the 'Play' button on the upper panel or select the cell and press Shift+Enter.

In [ ]:
%load_ext dotenv
%dotenv config.env

In [ ]:
%load_ext autoreload 
%autoreload 2

In [ ]:
# Import the dependencies.

import os
import pandas as pd
import shutil
from ipyfilechooser import FileChooser

from utils.set_widgets import mask_chooser, filter_data
from utils.tiff_writer import show_mask
from utils.conditions_to_dict import conditions_to_df, analysis_meta
from utils.csv_writer import update_db_analysis, show_df_style


In [ ]:
# Select the mask file generated in omnipose, .npy or .png
# .npy is preferred if available

im_path = os.getenv('SMPP_INPUT_DIR') # input folder
fc = FileChooser(im_path)
fc.filter_pattern = '*.npy'
display(fc)


In [ ]:
# Preview the mask file

m = show_mask(fc.selected_path, fc.selected_filename) 


In [ ]:
# Click 'Play' to search across the database and choose the timelapse ('Raw Images') 
# corresponding to the mask you have chosen.

# Use the 'clear' function in the 'date' field to remove any unwanted entry.
# It is ok to use a comma and space to separate the row numbers in the 'exclude' and 'include' fields.
# Keep all the fields intact to view the entire database (though this may become inconvenient soon)

database = pd.read_csv(os.getenv('SMPP_DATABASE'), sep = '\t', dtype = 'str')
fields_list, fld_widgets_list, box = filter_data(database)
display(box)


In [ ]:
# Run this cell when ready with the selection above,
# check if the file you choose is the one you want.

selected = conditions_to_df(database, fld_widgets_list, fields_list, 'Raw images')
selected


In [ ]:
# Run this cell to generate analysis-level idetifiers.
# You may want to add a free-form comment specific to the file (optional),
# otherwise, press 'enter'.

try:
    git_hash, uuid_analysis, comment, filename = analysis_meta(selected, entry_type='mask')
except ValueError:
    print(file_meta(selected))


In [ ]:
out_path = os.getenv('SMPP_DATA_DIR')+'/analysis/masks_omni/'
shutil.copyfile(fc.selected, out_path+filename)


In [ ]:
# Update the database, show the last 10 entries.
df = update_db_analysis(selected, comment, git_hash, uuid_analysis, out_path+filename, type='mask')
show_df_style(df.tail(10))


In [ ]:
# To upload a new mask, press 'Kernel' -> 'Restart Kernel'. 
# Otherwise, you can close the add_analysis_mask notebook, or shut down the interactive session.
